In [ ]:
# Preparing patch data
import rasterio
import numpy as np

with rasterio.open("S2_Prosopis_CNN_Stack.tif") as src:
    image = src.read() # (bands, height, width)
    profile = src.profile # save metadata

# with rasterio.open("S2_patchCNN_labels.tif") as src:
#     labels = src.read(1)
with rasterio.open("prosopis_mask.tif") as src:
    labels = src.read(1)

In [ ]:
# Preprocessing
# Normalize
image = image.astype(np.float32)

# Move bands to last dimension (CNN format)
image = np.moveaxis(image, 0, -1)   # (H, W, bands)

# Expand label dimension
labels = labels[..., np.newaxis]

In [ ]:
# Patch extraction 
# def extract_patches(image, labels, size=32, stride=32):
#     X, y = [], []
#     H, W, _ = image.shape

#     for i in range(0, H-size+1, stride):
#         for j in range(0, W-size+1, stride):
#             img = image[i:i+size, j:j+size]
#             lbl = labels[i:i+size, j:j+size]

#             if np.sum(lbl) == 0:
#                 if np.random.rand() > 0.1:
#                     continue

#             X.append(img)
#             y.append(int(np.mean(lbl) > 0))

#     return np.array(X), np.array(y)
import numpy as np

def extract_patches(image, labels, size=32, stride=32):

    X = []
    y = []

    h, w = labels.shape

    for i in range(0, h - size + 1, stride):
        for j in range(0, w - size + 1, stride):

            img_patch = image[i:i+size, j:j+size]
            lbl_patch = labels[i:i+size, j:j+size]

            # Skip incomplete patches
            if img_patch.shape[:2] != (size, size):
                continue

            if lbl_patch.shape != (size, size):
                continue

            # Skip empty patches
            if lbl_patch.size == 0:
                continue

            # Binary tile label
            label = 1 if np.mean(lbl_patch) > 0.01 else 0

            X.append(img_patch)
            y.append(label)

    return np.array(X), np.array(y)

In [ ]:
labels = labels.squeeze()
print("Labels shape:", labels.shape)

In [ ]:
# Create dataset 
X, y = extract_patches(image, labels, size=32, stride=32)

print(X.shape)  # (N, 32, 32, bands)
print(y.shape)  # (N,)

In [ ]:
# Train-validation split
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Defining the patch-based CNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    Conv2D(32, 3, activation='relu', input_shape=(32,32,X.shape[-1])),
    MaxPooling2D(),
    Conv2D(64, 3, activation='relu'),
    MaxPooling2D(),
    Conv2D(128, 3, activation='relu'),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

In [ ]:
# Compile model
model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train the CNN
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32
)

In [ ]:
# Training visualization
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()
plt.title("Training Loss")
plt.show()
plt.savefig("training_loss.png")

In [ ]:
# Accuracy assessment
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_val) > 0.5).astype(int)

print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

In [ ]:
# ROC curve
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(y_val, model.predict(X_val))
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.legend()
plt.show()
#plt.savefig("roc_auc.png")

In [ ]:
# Full image prediction
H, W, _ = image.shape
prob_map = np.zeros((H,W))
count = np.zeros((H,W))

for i in range(0, H-32+1, 32):
    for j in range(0, W-32+1, 32):
        patch = image[i:i+32, j:j+32]
        patch = patch[np.newaxis, ...]

        p = model.predict(patch, verbose=0)[0][0]
        prob_map[i:i+32, j:j+32] += p
        count[i:i+32, j:j+32] += 1

prob_map /= np.maximum(count, 1)

In [ ]:
# Thresholding
classified = (prob_map > 0.25).astype(np.uint8)

In [ ]:
# Save GeoTiff
profile.update(dtype=rasterio.uint8, count=1)

with rasterio.open("cnn_classification.tif", "w", **profile) as dst:
    dst.write(classified, 1)

In [ ]:
# Visualization
plt.imshow(prob_map, cmap='YlGn')
plt.colorbar(label='Prosopis probability')
plt.show()

plt.imshow(classified, cmap='Greens') 
plt.title("CNN Prosopis Map")
plt.show()
#plt.savefig("cnn_prosprob.png")

## Threshold F1 curve 

In [ ]:
y_val_prob = model.predict(X_val).ravel()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_score, recall_score

# Define threshold range
thresholds = np.linspace(0.01, 0.99, 50)

f1_scores = []
precision_scores = []
recall_scores = []

for t in thresholds:
    y_pred = (y_val_prob >= t).astype(int)

    f1_scores.append(f1_score(y_val, y_pred, zero_division=0))
    precision_scores.append(precision_score(y_val, y_pred, zero_division=0))
    recall_scores.append(recall_score(y_val, y_pred, zero_division=0))

In [ ]:
plt.figure(figsize=(7,5))
plt.plot(thresholds, f1_scores, label='F1-score', linewidth=2)
plt.plot(thresholds, precision_scores, '--', label='Precision')
plt.plot(thresholds, recall_scores, '--', label='Recall')

# Best threshold
best_idx = np.argmax(f1_scores)
best_t = thresholds[best_idx]

plt.axvline(best_t, color='red', linestyle=':', label=f'Best threshold = {best_t:.2f}')

plt.xlabel('Probability threshold')
plt.ylabel('Score')
#plt.title('Threshold–Performance Curve')
plt.legend()
plt.grid(True)
plt.savefig("threshold_performance_curve.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"Optimal threshold (max F1): {best_t:.3f}")

## RANDOM FOREST

In [ ]:
# Imports
import numpy as np
import rasterio
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_recall_curve, roc_curve, auc,
    f1_score, confusion_matrix
)

In [ ]:
import rasterio
import numpy as np

# Load image (Sentinel-2 composite)
with rasterio.open("S2_Prosopis_CNN_Stack.tif") as src:
    image = src.read().transpose(1, 2, 0)
    
# Load mask (Prosopis labels)
with rasterio.open("prosopis_mask.tif") as src:
    prosopis_mask = src.read(1)

print(image.shape, prosopis_mask.shape)
print("Unique values:", np.unique(prosopis_mask))
print("Positive %:", np.mean(prosopis_mask) * 100)

In [ ]:
# Normalize data
image = image / 10000.0  # Sentinel-2 scaling

In [ ]:
# Tile extraction
def extract_tiles(image, prosopis_mask, tile_size=128):
    X, y = [], []
    h, w = image.shape[:2]

    for i in range(0, h - tile_size, tile_size):
        for j in range(0, w - tile_size, tile_size):

            tile = image[i:i+tile_size, j:j+tile_size]
            label = prosopis_mask[i:i+tile_size, j:j+tile_size]

            if tile.shape[:2] != (tile_size, tile_size):
                continue

            # Presence-based labeling
            tile_label = 1 if np.sum(label) > 0 else 0

            X.append(tile)
            y.append(tile_label)

    return np.array(X), np.array(y)


X, y = extract_tiles(image, prosopis_mask)

print("Tiles:", X.shape)
print("Labels:", y.shape)

In [ ]:
# Balance Dataset
pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]

n_pos = len(pos_idx)

# Downsample background
neg_idx_sample = np.random.choice(neg_idx, size=n_pos*2, replace=False)

idx = np.concatenate([pos_idx, neg_idx_sample])

X = X[idx]
y = y[idx]

print("Balanced:", np.bincount(y))

In [ ]:
# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_val.shape)

## CNN MODEL

In [ ]:
def build_cnn(input_shape):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = tf.keras.layers.Dense(64, activation='relu')(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs)


model = build_cnn(X_train.shape[1:])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Train CNN
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=8,
    verbose=1
)

In [ ]:
# Train RF
# Flatten for RF
X_train_rf = X_train.reshape(X_train.shape[0], -1)
X_val_rf   = X_val.reshape(X_val.shape[0], -1)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_rf, y_train)

In [ ]:
# Predictions
# CNN
y_prob_cnn = model.predict(X_val).ravel()

# RF
y_prob_rf = rf.predict_proba(X_val_rf)[:, 1]

### Combined Evaluation Figure (ROC+PR+Threshold)

In [ ]:
# ROC
fpr_rf, tpr_rf, _ = roc_curve(y_val, y_prob_rf)
fpr_cnn, tpr_cnn, _ = roc_curve(y_val, y_prob_cnn)

auc_rf = auc(fpr_rf, tpr_rf)
auc_cnn = auc(fpr_cnn, tpr_cnn)

# PR
prec_rf, rec_rf, _ = precision_recall_curve(y_val, y_prob_rf)
prec_cnn, rec_cnn, _ = precision_recall_curve(y_val, y_prob_cnn)

# Threshold-F1
thresholds = np.linspace(0.01, 0.99, 50)
f1_rf = [f1_score(y_val, (y_prob_rf >= t).astype(int)) for t in thresholds]
f1_cnn = [f1_score(y_val, (y_prob_cnn >= t).astype(int)) for t in thresholds]

In [ ]:
# Plot Combined Figure
plt.figure(figsize=(15,4))

# ROC
plt.subplot(1,3,1)
plt.plot(fpr_rf, tpr_rf, label=f'RF AUC={auc_rf:.2f}')
plt.plot(fpr_cnn, tpr_cnn, label=f'CNN AUC={auc_cnn:.2f}')
plt.plot([0,1],[0,1],'k--')
plt.title("ROC")
plt.legend()

# PR
plt.subplot(1,3,2)
plt.plot(rec_rf, prec_rf, label='RF')
plt.plot(rec_cnn, prec_cnn, label='CNN')
plt.title("Precision-Recall")
plt.legend()

# Threshold
plt.subplot(1,3,3)
plt.plot(thresholds, f1_rf, label='RF')
plt.plot(thresholds, f1_cnn, label='CNN')
plt.title("F1 vs Threshold")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix at best threshold
best_t = thresholds[np.argmax(f1_cnn)]

y_pred = (y_prob_cnn >= best_t).astype(int)

cm = confusion_matrix(y_val, y_pred)
print(cm)

## Generate Classification Maps

In [ ]:
# Extract tiles with positions
def extract_tiles_with_pos(image, tile_size=128):
    tiles = []
    positions = []

    h, w = image.shape[:2]

    for i in range(0, h - tile_size, tile_size):
        for j in range(0, w - tile_size, tile_size):

            tile = image[i:i+tile_size, j:j+tile_size]

            if tile.shape[:2] != (tile_size, tile_size):
                continue

            tiles.append(tile)
            positions.append((i, j))

    return np.array(tiles), positions

In [ ]:
# Extract full tiles
X_full, positions = extract_tiles_with_pos(image)

print("Tiles:", X_full.shape)

In [ ]:
# Predict probabilities CNN
y_prob_full = model.predict(X_full).ravel()

In [ ]:
# Reconstruction probability maps
prob_map = np.zeros(image.shape[:2])

tile_size = 128

for (i, j), p in zip(positions, y_prob_full):
    prob_map[i:i+tile_size, j:j+tile_size] = p

In [ ]:
# Apply optimal threshold
best_t = thresholds[np.argmax(f1_cnn)]

binary_map = (prob_map >= best_t).astype('uint8')

In [ ]:
# Save GeoTiff
with rasterio.open("S2_Prosopis_CNN_Stack.tif") as src:
    profile = src.profile

profile.update(count=1, dtype='uint8')

with rasterio.open("prosopis_map.tif", "w", **profile) as dst:
    dst.write(binary_map, 1)

In [ ]:
# Save probability map
profile.update(dtype='float32')

with rasterio.open("prosopis_probability.tif", "w", **profile) as dst:
    dst.write(prob_map.astype('float32'), 1)

In [ ]:
# Visualize map
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(prob_map, cmap='viridis')
plt.title("Probability Map")

plt.subplot(1,2,2)
plt.imshow(binary_map, cmap='gray')
plt.title("Binary Map")

plt.show()

In [ ]:
# RF Map
X_full_rf = X_full.reshape(X_full.shape[0], -1)

y_prob_rf_full = rf.predict_proba(X_full_rf)[:, 1]

rf_map = np.zeros(image.shape[:2])

for (i, j), p in zip(positions, y_prob_rf_full):
    rf_map[i:i+tile_size, j:j+tile_size] = p

In [ ]:
# Add scale bar - 500m scale bar
# import matplotlib.pyplot as plt

# def add_scalebar(ax, pixel_size=10, length_m=500, location=(0.05, 0.05)):
#     """
#     pixel_size: meters per pixel (Sentinel-2 = 10)
#     length_m: scale bar length in meters
#     location: (x, y) in axis fraction
#     """
#     length_pixels = int(length_m / pixel_size)

#     xlim = ax.get_xlim()
#     ylim = ax.get_ylim()

#     x_start = xlim[0] + location[0] * (xlim[1] - xlim[0])
#     y_start = ylim[0] + location[1] * (ylim[1] - ylim[0])

#     ax.plot([x_start, x_start + length_pixels],
#             [y_start, y_start],
#             linewidth=3)

#     ax.text(x_start + length_pixels / 2, y_start,
#             f'{length_m} m',
#             ha='center', va='bottom')

In [ ]:
# Add North Arrow
# def add_north_arrow(ax, location=(0.9, 0.1)):
#     xlim = ax.get_xlim()
#     ylim = ax.get_ylim()

#     x = xlim[0] + location[0] * (xlim[1] - xlim[0])
#     y = ylim[0] + location[1] * (ylim[1] - ylim[0])

#     ax.annotate(
#         'N',
#         xy=(x, y),
#         xytext=(x, y - 50),
#         arrowprops=dict(facecolor='black', width=3, headwidth=10),
#         ha='center'
#     )

In [ ]:
# Plot comparison Figure
import matplotlib.pyplot as plt
vmin = 0
vmax = 1

plt.figure(figsize=(15,5))
# fig, axes = plt.subplots(1, 2, figsize=(12,6))
#RF
# axes[0].imshow(rf_map, vmin=0, vmax=1)
# axes[0].set_title("RF Probability")
# add_scalebar(axes[0])
# add_north_arrow(axes[0])
# axes[0].axis('off')
plt.subplot(1,3,1)
plt.imshow(rf_map, vmin=vmin, vmax=vmax)
plt.title("RF Probability")
plt.colorbar()

# CNN
# axes[1].imshow(prob_map, vmin=0, vmax=1)
# axes[1].set_title("CNN Probability")
# add_scalebar(axes[1])
# add_north_arrow(axes[1])
# axes[1].axis('off')
plt.subplot(1,3,2)
plt.imshow(prob_map, vmin=vmin, vmax=vmax)
plt.title("CNN Probability")
plt.colorbar()

plt.subplot(1,3,3)
plt.imshow(binary_map, cmap='gray')
plt.title("CNN Binary")

plt.tight_layout()
plt.show()

In [ ]:
# Zoom into area of interest
# Example crop
crop = slice(1000, 1500)

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(rf_map[crop, crop], vmin=0, vmax=1)
plt.title("RF (Zoom)")
plt.colorbar()

plt.subplot(1,3,2)
plt.imshow(prob_map[crop, crop], vmin=0, vmax=1)
plt.title("CNN (Zoom)")
plt.colorbar()

plt.subplot(1,3,3)
plt.imshow(binary_map[crop, crop], cmap='gray')
plt.title("CNN Binary (Zoom)")

plt.tight_layout()
plt.show()

In [ ]:
# Overlay detection on RGB
rgb = image[:, :, [3,2,1]]  # adjust bands if needed

plt.figure(figsize=(6,6))
plt.imshow(rgb)
plt.imshow(binary_map, alpha=0.4, cmap='Reds')
plt.title("CNN Detection Overlay")
plt.show()

In [ ]:
print("Prob min/max:", prob_map.min(), prob_map.max())
print("Unique binary:", np.unique(binary_map))

In [ ]:
print("Unique mask:", np.unique(prosopis_mask))

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])

In [ ]:
plt.hist(y_prob_cnn, bins=50)
plt.title("CNN probabilities")
plt.show()

In [ ]:
binary_map = (prob_map > 0.01).astype('uint8')

In [ ]:
print("Best threshold:", best_t)